# Studio · Compute로 경험하는 MLOps
**좋은 모델만 등록·배포하고, 새 모델로 전환했다가 이전 모델로 되돌립니다.**

먼저 [학습자 시작 안내](../docs/learner-start.md)에서 `준비 완료`를 확인하세요. 여기서 새 RG나 네트워크를 만들지 않습니다.

**Run all 금지.** `Shift+Enter`로 한 셀씩 실행하고 각 단계의 완료 조건을 확인합니다. Job/Endpoint 화면은 **새 탭**으로 여세요. 실행 중 같은 탭에서 다른 Workspace 메뉴로 이동하거나 kernel/Compute를 바꾸지 않습니다.

목차(Table of contents)를 열면 00–07로 이동할 수 있습니다. **03의 두 오류만 의도된 실패**이며, 04부터 추가 추론 VM 비용이 발생합니다.

## 00 · 준비 확인
**할 일:** 아래 셀로 Python·프로젝트·설정·로그인 계정을 확인합니다.

**Studio에서 확인:** Compute는 본인 Instance, kernel은 **AML MLOps Lab (Python 3.12)**입니다.

**완료 조건:** `준비 완료`와 의도한 Workspace가 출력됩니다. **LAB_ID를 기록**하세요. 다시 연결할 때만 `RESUME_LAB_ID`에 그 값을 넣고 00을 실행합니다. 기존 submit/배포 생성 셀은 다시 실행하지 않습니다.

In [ ]:
import os
import re
import sys
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4

if sys.version_info[:2] != (3, 12):
    raise RuntimeError('AML MLOps Lab (Python 3.12) kernel을 선택한 뒤 00을 다시 실행하세요.')
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), None)
if root is None or not (root / 'mlops_lab').is_dir():
    raise RuntimeError('프로젝트 전체 폴더 안의 Notebook을 여세요. learner-start.md의 준비 B를 확인하세요.')
if not (root / 'config.json').is_file():
    raise FileNotFoundError('프로젝트 루트의 config.json이 없습니다. 준비 C를 완료하세요.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
from mlops_lab.config import Settings
from mlops_lab.data import make_dataset
from mlops_lab.pipeline import register_assets
from mlops_lab.operations import (
    submit, wait_for_run, snapshot_run, download_report, register_model,
    deploy, wait_for_deployment, set_traffic, invoke, cleanup_runtime,
)

RESUME_LAB_ID = ''
if RESUME_LAB_ID and not re.fullmatch(r'[0-9]{14}(?:-[a-f0-9]{6})?', RESUME_LAB_ID):
    raise ValueError('RESUME_LAB_ID에는 이전에 출력된 LAB_ID를 그대로 넣으세요.')
settings = Settings.load()
client = settings.client()
lab_id = RESUME_LAB_ID or (datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S') + '-' + uuid4().hex[:6])
baseline_run, bad_run, retrain_run = [f'{name}-{lab_id}' for name in ('baseline', 'bad', 'retrain')]
version1, version2, rejected_version = [lab_id + suffix for suffix in ('1', '2', '9')]
print('준비 완료:', settings.expected_account, settings.workspace)
print('학습 Compute:', settings.compute_cluster)
print('기록할 LAB_ID:', lab_id)
print('이어서 할 때 RESUME_LAB_ID =', repr(lab_id))

## 01 · 자산 등록
**할 일:** 합성 데이터 v1/v2, 재사용할 처리 단계 3개, 고정 환경 버전을 등록합니다.

**Studio에서 확인:** **Data → mlops-synthetic-regression**, **Components → mlops_prepare / mlops_train / mlops_evaluate**, **Environments → Curated → sklearn-1.5:53**.

**완료 조건:** v1은 학습 800행·검증 200행, v2는 학습 1,200행·검증 200행입니다. 검증 데이터는 같고 개인정보가 없는 합성 데이터입니다. `row_id`/`partition`은 모델 입력이 아닙니다.

In [ ]:
assets = register_assets(client, settings)
print('환경:', assets['environment'])
print('컴포넌트 버전:', settings.component_version)
print({v: {'학습': d['training_rows'], '검증': d['validation_rows']} for v, d in assets['data'].items()})
assert assets['data']['1']['validation_sha256'] == assets['data']['2']['validation_sha256'], '두 버전의 검증 데이터가 다릅니다.'
make_dataset('1').head()

## 02 · 학습
**할 일:** 첫 셀로 제출하고 다음 셀로 기다립니다. 제출은 Instance에서, 실제 처리는 Cluster에서 실행됩니다. **RMSE는 작을수록 좋은 예측 오차**입니다.

**Studio에서 확인:** 출력 URL을 **새 탭**으로 열어 **evaluate_gate → Metrics**를 봅니다. 모델 파일은 **train_model → Outputs + logs**에 있습니다.

**완료 조건:** `Completed`, `approved=true`, **RMSE ≤ 3**(예상 약 1.84). `Queued`/`Running`/timeout이면 submit을 반복하지 말고 대기 셀만 다시 실행합니다.

In [ ]:
baseline = submit(client, settings, baseline_run, '1', alpha=1.0, max_rmse=3.0)
print('새 탭에서 열기:', baseline['studio_url'])

In [ ]:
wait_for_run(client, settings, baseline_run)
baseline_report = download_report(client, settings, baseline_run)
assert baseline_report['approved'] is True and baseline_report['rmse'] <= 3.0, '02 미완료: 평가 보고서를 확인하세요.'
baseline_report

## 03 · 품질 게이트
**할 일:** 성능 미달 모델을 제출하고 대기합니다. 그 다음 확인 셀로 실패 원인을 판정합니다.

**Studio에서 확인:** **Jobs → 해당 실행 → evaluate_gate → Metrics / Outputs + logs**, 이후 **Models**.

**완료 조건:** `Failed` + 실패 단계 `evaluate_gate` + `approved=false` + **RMSE > 3**(예상 약 23.4), 그리고 등록 차단입니다. **두 오류를 확인한 뒤 04로 이동**합니다. 다른 단계/인증/네트워크 오류면 멈추고 [판단표](../docs/learner-start.md#기다릴지-고칠지-판단하기)를 따르세요.

In [ ]:
bad = submit(client, settings, bad_run, '1', alpha=1000000.0, max_rmse=3.0)
print('새 탭에서 열기:', bad['studio_url'])

### 03-A · 의도된 오류 1/2
다음 대기 셀의 **RuntimeError가 예상 결과**입니다. 오류 후 바로 아래의 확인 셀을 실행합니다. 모든 빨간 오류를 정상으로 간주하지 않습니다.

In [ ]:
wait_for_run(client, settings, bad_run)

In [ ]:
bad_state = snapshot_run(client, settings, bad_run)
bad_report = download_report(client, settings, bad_run)
assert bad_state['status'] == 'Failed', '예상한 실패가 아닙니다.'
assert any(c['display_name'] == 'evaluate_gate' and c['status'] == 'Failed' for c in bad_state['children']), '평가가 아닌 다른 단계의 오류입니다. 원인을 해결하세요.'
assert bad_report['approved'] is False and bad_report['rmse'] > 3.0, '품질 게이트 실패 조건과 다릅니다.'
bad_report

### 03-B · 의도된 오류 2/2
다음 셀은 **Registration blocked** 오류가 나야 합니다. Studio의 Models에 출력한 거절 버전이 없는 것을 확인한 뒤 **04로 진행**합니다.

In [ ]:
print('등록되면 안 되는 버전:', rejected_version)
register_model(client, settings, bad_run, rejected_version)

## 04 · blue 배포
**할 일:** 02에서 통과한 모델을 Registry(모델 보관소)에 등록하고 `blue`(기존 모델)에 배포합니다. **여기부터 추가 추론 VM 비용이 발생합니다.**

**Studio에서 확인:** **Models → mlops-ridge → 해당 버전**의 `source_job` / `quality_gate`; **Endpoints → 해당 endpoint → blue**의 상태.

**완료 조건:** 모델 원본 Job 연결, blue `Succeeded`, 직접/기본 호출 모두 **5행 → 숫자 5개**입니다. `Submitted`/`Creating`/`ready=false`는 아직 완료가 아닙니다. 배포를 다시 만들지 말고 대기 셀만 재실행합니다.

In [ ]:
registered1 = register_model(client, settings, baseline_run, version1)
print(registered1['id'])

In [ ]:
deploy(client, settings, version1, 'blue', wait=False)

In [ ]:
wait_for_deployment(client, settings, 'blue')

In [ ]:
set_traffic(client, settings, 'blue')

In [ ]:
blue_direct = invoke(client, settings, 'blue')
blue_routed = invoke(client, settings)
np.testing.assert_allclose(blue_routed['predictions'], blue_direct['predictions'], rtol=1e-6, atol=1e-6)
blue_routed

## 05 · 재학습
**할 일:** 학습 행이 400개 추가된 데이터 v2로 같은 파이프라인을 실행하고 새 모델 버전을 등록합니다.

**Studio에서 확인:** 두 실행의 **evaluate_gate → Metrics**, **Models**의 서로 다른 버전과 원본 Job.

**완료 조건:** 새 실행 `Completed`, RMSE ≤ 3(예상 약 1.82), `validation_sha256` 일치, 새 버전 등록입니다. 같은 검증 데이터를 비교하며 사람이 06의 전환을 결정합니다.

In [ ]:
retrain = submit(client, settings, retrain_run, '2', alpha=0.1, max_rmse=3.0)
print('새 탭에서 열기:', retrain['studio_url'])

In [ ]:
wait_for_run(client, settings, retrain_run)
baseline_report = download_report(client, settings, baseline_run)
retrain_report = download_report(client, settings, retrain_run)
assert retrain_report['validation_sha256'] == baseline_report['validation_sha256'], '검증 데이터가 달라 직접 비교할 수 없습니다.'
assert retrain_report['approved'] is True and retrain_report['rmse'] <= 3.0, '재학습 모델이 품질 기준을 통과하지 못했습니다.'
print({'baseline_rmse': baseline_report['rmse'], 'retrain_rmse': retrain_report['rmse']})

In [ ]:
registered2 = register_model(client, settings, retrain_run, version2)
print(registered2['id'])

## 06 · 전환·롤백
**할 일:** `green`(새 모델)을 직접 확인한 뒤 기본 경로를 green으로 전환하고 blue로 롤백합니다.

**Studio에서 확인:** **Endpoints → 해당 endpoint → traffic**. 직접 호출은 traffic 설정을 우회합니다.

**완료 조건:** 기본 경로의 응답이 **blue → green → blue**로 확인됩니다. 비교가 실패하면 상태를 확인하고 해당 **호출·비교 셀만** 다시 실행하세요. 다시 배포하거나 전체 실습을 반복하지 않습니다.

In [ ]:
deploy(client, settings, version2, 'green', wait=False)

In [ ]:
wait_for_deployment(client, settings, 'green')
green_direct = invoke(client, settings, 'green')
green_direct

In [ ]:
set_traffic(client, settings, 'green')

In [ ]:
green_direct = invoke(client, settings, 'green')
green_routed = invoke(client, settings)
np.testing.assert_allclose(green_routed['predictions'], green_direct['predictions'], rtol=1e-6, atol=1e-6)
green_routed

In [ ]:
set_traffic(client, settings, 'blue')

In [ ]:
blue_direct = invoke(client, settings, 'blue')
rollback_routed = invoke(client, settings)
np.testing.assert_allclose(rollback_routed['predictions'], blue_direct['predictions'], rtol=1e-6, atol=1e-6)
rollback_routed

## 07 · 비용 정리
**할 일:** endpoint를 삭제하고 Instance를 중지합니다. 중도 종료라면 먼저 [실행 중 Job 취소](../docs/learner-start.md#중간에-그만둘-때)를 수행합니다.

**Studio에서 확인:** **Endpoints**, **Compute → Compute instances / Compute clusters**.

**완료 조건:** endpoint 없음, Instance **Stopped**, Cluster **실제 0노드**. Instance 중지로 연결이 끊길 수 있으므로 확인은 Studio에서 마칩니다.

최소 노드 0 설정만으로 종료를 판단하지 않습니다. 마지막 작업 후 idle 축소를 기다립니다. Premium ACR·Storage·디스크·Private Endpoint 비용은 남으며, 전체 RG 삭제는 강사와 보관 필요성을 확인한 뒤 [정리 문서](../docs/troubleshooting.md#전체-환경이-더-이상-필요하지-않은-경우)를 따릅니다.

In [ ]:
cleanup_runtime(client, settings, delete_endpoint=True)